# Organizational Culture Assessment Instrument (OCAI)
### High-Performance Compositional Data Analysis (CoDa) Engine with Polars

This notebook demonstrates the complete OCAI analytical pipeline using the **`ocai`** library:
1. **Multiplicative Zero Replacement** on the simplex (base 100).
2. **Closed Geometric Mean** for Collective Full Profile (**CFP**) and Collective Dimensional Profile (**CDP**).
3. **Simplex Block Bootstrap** (10,000 replicates) with Aitchison confidence regions (95%).
4. **Paired Dimension-Level Permutation Testing** ($H_0: \text{Now} = \text{Preferred}$).
5. **Hierarchical Sub-Group / Departmental Analysis** with configurable `minGroupSize` and homonymous sub-departments.
6. **Individualized Department Results & Multi-Format Exports** (Excel multi-sheet, CSV, JSON).
7. **Competing Values Framework (CVF)** Radar Visualizations.

In [ ]:
%load_ext autoreload
%autoreload 2

from ocai import OcaiAnalyzer
import matplotlib.pyplot as plt

# Initialize and execute the analytical pipeline
analyzer = OcaiAnalyzer(
    filePath="exampledb.csv",
    separator=";",
    groupBy=["department", "subdepartment"],  # Hierarchical composite grouping
    minGroupSize=5,                            # Configurable minimum sample threshold
    numSamples=10000,                          # Bootstrap replicates
    numPermut=5000,                            # Permutations
    alpha=0.05                                 # Significance level (95% CI)
)

analyzer.run()

## 1. Executive Summary & Interactive HTML Report
The summary below is rendered directly as a styled HTML report (without console prints), suitable for interactive inspection in Jupyter, while `.to_dict()` provides raw JSON for AWS Lambda.

In [ ]:
# Returning the report object in the last line invokes its native rich HTML representation
analyzer.summary()

## 2. Individualized Departmental Results (CFP & CDP)
Access detailed confidence intervals and dimensional permutation results for each department individually as Polars DataFrames.

In [ ]:
# 1. Confidence intervals (CFP) for ALL departments
dfAllCfp = analyzer.getAllGroupsConfidenceIntervals()
dfAllCfp

In [ ]:
# 2. Dimensional hypothesis testing (CDP) for ALL departments
dfAllCdp = analyzer.getAllGroupsDimensionalResults()
dfAllCdp

## 3. Exporting Individualized Results (Excel, CSV, and JSON)
Export all findings into a multi-sheet Excel workbook, individual CSV files per department, or a structured JSON payload for cloud dashboards.

In [ ]:
# Export multi-sheet Excel workbook (Benchmarking, Departments_CFP, Departments_CDP, Overall)
excelPath = analyzer.exportToExcel("Results/ocai_complete_results.xlsx")
print(f"Excel workbook exported to: {excelPath}")

# Export CSVs (benchmarking, all departments, and individual department files)
csvFiles = analyzer.exportToCsv("Results")
print(f"Created {len(csvFiles)} CSV files in 'Results/' directory.")

# Export JSON summary (for AWS Lambda / S3 storage)
jsonPath = analyzer.exportToJson("Results/ocai_summary.json")
print(f"JSON report exported to: {jsonPath}")

## 4. Visualizations: Competing Values Framework (CVF) Radars
Polar radar charts displaying Current vs. Preferred culture, sub-group overlay benchmarking, and 6-dimension panels.

In [ ]:
# 1. Global Organizational Profile (CFP) with 95% Confidence Intervals
figGlobal = analyzer.plotGlobal(figsize=(7, 7))
plt.show()

In [ ]:
# 2. Sub-Group Cultural Overlay (Benchmarking departments on a single radar)
figSub = analyzer.plotSubGroupsComparison(state="Now", figsize=(8, 8))
plt.show()

In [ ]:
# 3. The 6 Individual Dimensions (CDP)
figDims = analyzer.plotDimensions(figsize=(16, 11))
plt.show()